## Context Aware & Stateful Tools

### We'll use a E-commerce Customer Support AI Agent example to understand need of ToolRuntime & How to Use It.

**Scenario:** : A customer asks:
 >   "What's the status of my order 12345?"

The agent has multiple tools:
* get_order_details
* cancel_order
* send_email

We'll use this as example to understand ToolRuntime concept.

#### Before goint through example usecases, it's worth noting some information arounf Agent s and Langchain.
**Langchain Provides 2 Reserved arguments for Tools**
| Parameter name | Purpose |
|----------|---------------|
| config  | Reserved for passing RunnableConfig to tools internally |
| runtime: ToolRuntime | Reserved for ToolRuntime parameter (accessing state, context, store) |


#### Information in `ToolRuntime` will NOT BE VISIBLE TO LLMs**
* So if we have some secret information or user_ids etc, we can use ToolRuntime to hide that information.
* Prove the hiding claim in below code block --> `runtime` will NOT appear here, only `action` will be visible


In [1]:
from langchain.tools import tool, ToolRuntime

@tool
def get_last_order_details(action: str, runtime: ToolRuntime) -> str:
    """Find the last order details of the customer in this conversation."""
    return "No orders found."

print("Tool schema seen by the model:", get_last_order_details.args)

Tool schema seen by the model: {'action': {'title': 'Action', 'type': 'string'}}


### ToolRuntime: provides below Runtime information
```python
    def get_orders(runtime: ToolRuntime):
        runtime.state  # Short-term memory - mutable data: Access conversation history, track tool call counts
        runtime.store  # Long-term memory - Save user preferences, maintain knowledge base
        runtime.context # Personalize responses based on user identity e.g runtime.context.user_id
        runtime.config # Access callbacks, tags, and metadata
        runtime.stream_writer # Emit real-time updates during tool execution
        runtime.execution_info # Process and retry information for the current execution (thread ID, run ID, attempt number)
        runtime.tool_call_id # 
```

### In Langchain there are 2 Ways LLMs see the exiting tools
* `model.bind_tools([list_of_tools])` This only makes the model **AWARE** which tools exist — it **Does Not RUN** anything.
* Langchain's Agent **`create_agent`** this actually runs the tool and looping back for a final answer.

<img src="../../assets/bind_vs_create_agent.png" width="1200" height="150">


In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import BaseMessage, HumanMessage
from typing import TypedDict, Sequence,Annotated
from pydantic import BaseModel
import operator

import sys
sys.path.append('..')
from utils.helper import pretty_print_messages

load_dotenv()

True

## 1. Access State - Mutable Short Term Memory: `runtime.state`

In [3]:

# 1. Define the Custom Agent State structure
class CustomAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    user_membership_tier: str  # <--- Our custom state field

# 2. Define the tool using ToolRuntime
@tool()
def apply_user_discount(item_id: str, runtime: ToolRuntime) -> str:
    """Calculate an item's price by looking up the user's active membership tier."""
    # Pull the custom state parameter out of the unified runtime object
    current_state = runtime.state
    user_tier = current_state.get("user_membership_tier", "guest")
    
    base_price = 100.0
    discount = 0.20 if user_tier == "premium" else 0.0
    final_price = base_price * (1 - discount)
    
    return f"Item {item_id} final price for {user_tier} tier: ${final_price:.2f}"

# 4. Create the Agent using create_agent
stateful_agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[apply_user_discount],
    state_schema=CustomAgentState
)

In [4]:

premium_input = {
        "messages": [HumanMessage(content="How much does item 'xyz-789' cost for me?")],
        "user_membership_tier": "premium"
    }
normal_input = {
        "messages": [HumanMessage(content="How much does item 'xyz-789' cost for me?")],
        "user_membership_tier": "standard"
    }

In [5]:
result_normal_user = stateful_agent.invoke(normal_input)
pretty_print_messages(result_normal_user)

Message 1
Role : Human
--------------------------------------------------------------------------------
How much does item 'xyz-789' cost for me?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : apply_user_discount
    Args : {'item_id': 'xyz-789'}
    ID   : call_B4rKKH9tNeB1YjLsMEjSGRYC
Message 3
Role : Tool
--------------------------------------------------------------------------------
Item xyz-789 final price for standard tier: $100.00

Tool Name : apply_user_discount
Tool Call : call_B4rKKH9tNeB1YjLsMEjSGRYC
Message 4
Role : AI
--------------------------------------------------------------------------------
For item xyz-789, your current standard tier price is $100.00. 

Would you like me to check prices for a different tier or item?


In [6]:
result_premium_user = stateful_agent.invoke(premium_input)
pretty_print_messages(result_premium_user)

Message 1
Role : Human
--------------------------------------------------------------------------------
How much does item 'xyz-789' cost for me?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : apply_user_discount
    Args : {'item_id': 'xyz-789'}
    ID   : call_mk5oZDJLHwCuvgF3yGP26ZaZ
Message 3
Role : Tool
--------------------------------------------------------------------------------
Item xyz-789 final price for premium tier: $80.00

Tool Name : apply_user_discount
Tool Call : call_mk5oZDJLHwCuvgF3yGP26ZaZ
Message 4
Role : AI
--------------------------------------------------------------------------------
The item xyz-789 costs $80.00 for your Premium tier. This is the final price after your membership discount.

Would you like me to add it to your cart or show prices for other tiers?


## 2. Context - Static Run Configuration `runtime.context`

In [7]:
ORDERS_DATABASE = {
    "user_123": {
       "ORD001":
       { 
            "name": "Shivam",
            "account_type": "Premium",
            "order_status": 'Delivered',
            "total_amount": 5000
        },
        "ORD002":
        { 
            "name": "Shivam",
            "account_type": "Premium",
            "order_status": 'Placed',
            "total_amount": 1000,
            "estimated_delivery": "2024-06-15"
        }
    },
    
    "user_456": {
       "ORD005":
       {
            "name": "Satyam",
            "account_type": "Standard",
            "order_status": 'Placed',
            "total_amount": 1200,
            "estimated_delivery": "2025-04-15"
       }
    }
}

class UserContext(BaseModel):
    user_id: str

@tool
def get_order_status(runtime: ToolRuntime[UserContext], order_id: str) -> str:
    """Retrieve the order status for the user based on their user_id from the runtime context.
    Args:
        order_id (str): The ID of the order to retrieve the status for.
    """
    user_id = runtime.context.user_id
    if user_id in ORDERS_DATABASE and order_id in ORDERS_DATABASE[user_id]:
        order = ORDERS_DATABASE[user_id][order_id]
        order_info = f"Order ID: {order_id}\nName: {order['name']}\nType: {order['account_type']}\nOrder Status: {order['order_status']}\nTotal Amount: ${order['total_amount']}"
        if 'estimated_delivery' in order:
            order_info += f"\nEstimated Delivery: {order['estimated_delivery']}"
        return order_info
    return "User or Order not found"

contextual_agent = create_agent(
    model='openai:gpt-5-nano',
    tools=[get_order_status],
    context_schema=UserContext,
    system_prompt="You are a helpful assistant that provides order status information based on the user's context."
)


In [8]:
result = contextual_agent.invoke(
    {"messages": [HumanMessage(content="I have an order with order id ORD002. Can you tell me its status? and what is the estimated delivery date?")]},
    context=UserContext(user_id="user_123")
)

In [9]:
pretty_print_messages(result)

Message 1
Role : Human
--------------------------------------------------------------------------------
I have an order with order id ORD002. Can you tell me its status? and what is the estimated delivery date?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : get_order_status
    Args : {'order_id': 'ORD002'}
    ID   : call_krhcSTmw0bPrTNTsaQulSwZe
Message 3
Role : Tool
--------------------------------------------------------------------------------
Order ID: ORD002
Name: Shivam
Type: Premium
Order Status: Placed
Total Amount: $1000
Estimated Delivery: 2024-06-15

Tool Name : get_order_status
Tool Call : call_krhcSTmw0bPrTNTsaQulSwZe
Message 4
Role : AI
--------------------------------------------------------------------------------
Here are the details for ORD002:

- Status: Placed
- Estimated Delivery Date: 2024-06-15

Would you like me to track this further or set a reminder for updates?


## 3. Access Sore - Long-term memory: `runtime.store`
* Access persistent data across conversations using the **store**. 
* This allows you to save and retrieve user-specific or application-specific data. 
* Accessing / Remembering a Customer's Cart information Across Entirely Separate Visits using `runtime.store`
* `runtime.state` only covers Current conversation. 
* For memory that survives across completely separate sessions, a `Store` is attached to the agent and reached via `runtime.store`.
* For production deployments, use persistent store implementation like `PostgresStore` instead of `InMemoryStore`

### Example usecase [Milestone Tracker](03_milestone_tracker.ipynb) Notebook

In [10]:
from langgraph.store.memory import InMemoryStore

cart_store = InMemoryStore()

@tool
def store_item_to_cart(customer: str, item: str, runtime: ToolRuntime) -> str:
    """Save an item to a customer's cart for future visits."""
    # Fetch existing items
    result = runtime.store.get((customer, "preferences"), "items")
    items = result.value["value"] if result else []

    # Append new item
    items.append(item)

    # Store updated list
    runtime.store.put((customer, "preferences"), "items", {"value": items})
    return f"Got it -- I've added {item} to your cart."

@tool
def get_items_from_cart(customer: str, runtime: ToolRuntime) -> str:
    """Recall the items in a customer's cart, if we've saved them before."""
    result = runtime.store.get((customer, "preferences"), "items")
    return result.value["value"] if result else "We don't have any items saved for this customer yet."

memory_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[store_item_to_cart, get_items_from_cart],
    store=cart_store,
)

**Add Macbook to the Cart**

In [11]:
memory_agent.invoke({"messages": [("user", "Hi, I'm customer shivam, I want to add an item Macbook to my cart.")]})
result = memory_agent.invoke({"messages": [("user", "What items do I have in my cart? I'm shivam.")]})

In [79]:
print(result["messages"][-1].content)

Hi Shivam! You currently have the following item in your cart: Macbook.

Would you like to add more items, remove something, or proceed to checkout?


**Add Cricket Bat to the Cart and see items in Cart**

In [13]:
memory_agent.invoke({"messages": [HumanMessage(content="Hi, I'm customer shivam, can you add cricket bat in my cart.")]})
result = memory_agent.invoke({"messages": [HumanMessage(content="What items do I have in my cart? I'm shivam.")]})

In [82]:
print(result["messages"][-1].content)

Here are the items in your cart, Shivam:
- Macbook
- cricket bat

Would you like to remove anything, view details, or proceed to checkout? I can also add more items if you’d like.


In [14]:
pretty_print_messages(result)

Message 1
Role : Human
--------------------------------------------------------------------------------
What items do I have in my cart? I'm shivam.
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : get_items_from_cart
    Args : {'customer': 'shivam'}
    ID   : call_gAZuQ0eMyw2lkknFqOzBxVAE
Message 3
Role : Tool
--------------------------------------------------------------------------------
["Macbook", "cricket bat"]

Tool Name : get_items_from_cart
Tool Call : call_gAZuQ0eMyw2lkknFqOzBxVAE
Message 4
Role : AI
--------------------------------------------------------------------------------
You have the following items in your cart, Shivam:
- Macbook
- cricket bat

Would you like to add more items or proceed to checkout?


#### Inspecting the Store Directly: `.search()`
* Beyond `.get()` (fetch one specific key) and `.put()` (save one)
* `Store` also supports `.search()`that lists every item saved under a given namespace, without needing to know each exact key in advance. 
* Genuinely useful for debugging or building an admin view of what's been saved.

In [83]:
items = cart_store.search(("shivam", "preferences"))
for item in items:
    print(item)

Item(namespace=['shivam', 'preferences'], key='items', value={'value': ['Macbook', 'cricket bat']}, created_at='2026-08-02T05:47:45.711344+00:00', updated_at='2026-08-02T05:47:45.711347+00:00', score=None)


## 4. Two Stream Writer : `execution_info`
`execution_info` gives the current thread/run/retry identity. 
```python
    @tool
    def log_execution_context(runtime: ToolRuntime) -> str:
        """Log execution identity information."""
        info = runtime.execution_info
        print(f"Thread: {info.thread_id}, Run: {info.run_id}")
        print(f"Attempt: {info.node_attempt}")
        return "done"
```

In [85]:
@tool
def log_booking_context(runtime: ToolRuntime) -> str:
    """Log identity info about the current booking session -- useful for debugging."""
    info = runtime.execution_info
    print(f"  [debug] thread={info.thread_id}, run={info.run_id}, attempt={info.node_attempt}")
    server = runtime.server_info
    print(f"  [debug] server_info is None locally: {server is None}")
    return "Logged."

print("Tool defined. server_info being None locally is EXPECTED --")
print("it only populates once deployed to LangGraph Server, never during local development.")


Tool defined. server_info being None locally is EXPECTED --
it only populates once deployed to LangGraph Server, never during local development.


### Tool Return Values
1. **Return a string**
    * Return a string when the tool should provide plain text for the model and use in its next response.
    * The return value is converted to a ToolMessage.
    * The model sees that text and decides what to do next.

2. **Return an object**
    * Return an object (for example, a dict) when your tool produces structured data for model to inspect.
    * The object is serialized and sent back as tool output.
    * The model can read specific fields and reason over them.

3. **Return multimodal content**

    When the model supports multimodal tool results, the tool can return content blocks so the model receives text, images, and other media in one tool result
    ```python
        @tool
        def capture_screenshot() -> list[dict]:
            """Capture a screenshot of the current page."""
            return [
                {"type": "text", "text": "Screenshot of the current page:"},
                {"type": "image", "url": "https://example.com/page.png"},
            ]
    ```
    * The return value is converted to a ToolMessage with multimodal content.
    * Use `message.content_blocks` to read the block list after the tool runs.
    * Check model’s capabilities before returning images, audio, or video.

4. **Return directly from a tool `return_direct`**
    * Sometimes a tool's raw output IS the final answer — no rephrasing needed.
    * Use when the exact tool wording matters that must never be paraphrased.
    * Declare `@tool(return_direct=True)`, the agent returns the tool output directly without another LLM call. 
    * With this Agent's observation step will be skipped and Once this tool is called, response will directly be sent to user.
    * In this case Last message will be a `ToolMessage` instead of `AIMessage`

In [15]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Refunds are applicable till before the delivery date. No refunds after that."

direct_agent = create_agent(model="openai:gpt-5-nano", tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [HumanMessage(content="What's your refund policy?")]})

{'messages': [HumanMessage(content="What's your refund policy?", additional_kwargs={}, response_metadata={}, id='61d9f701-fb39-4a8f-ba7b-0421a6029746'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 125, 'total_tokens': 275, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E8RMbnwKzcA6noEWqKuh1Y4UEfL7J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fc2da-45e7-74b2-9d27-f4c85b67e0bb-0', tool_calls=[{'name': 'get_exact_refund_policy', 'args': {}, 'id': 'call_793Y42wjYiCenw3Vo8ekQ6ME', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

In [16]:
pretty_print_messages(result)

Message 1
Role : Human
--------------------------------------------------------------------------------
What's your refund policy?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : get_exact_refund_policy
    Args : {}
    ID   : call_793Y42wjYiCenw3Vo8ekQ6ME
Message 3
Role : Tool
--------------------------------------------------------------------------------
Refunds are applicable till before the delivery date. No refunds after that.

Tool Name : get_exact_refund_policy
Tool Call : call_793Y42wjYiCenw3Vo8ekQ6ME


**We get exact same message returned by Tool, there is No AI call further to refine it**

In [17]:
result['messages'][-1].content

'Refunds are applicable till before the delivery date. No refunds after that.'

## Dynamic tool selection
* Dynamic tools available to the agent, is modified at runtime rather than defined all upfront. 
* With dynamic tools, the set of tools available to the agent is modified at runtime rather than defined all upfront.

<img src="../../assets/many_tools.png" width="600" height="400">

* Suppose we have 300 tools and each tool has information of ~500 tokens
* Showing all available tools to LLM will send unnecessary tool tokens, in this case 300 * 500 = 1,50,000 Total tokens for tools information only will be sent to LLM in each user query.
* Another problem is our Model will also get confused, too many tools may overwhelm the model (overload context) and increase errors
* Dynamic tool selection enables adapting the available toolset based on
    * authentication state
    * user permissions etc.

#### Analogy: Access to premium Entertainment services
E-commerce have some services (Amazon Prime) which are applicable to only Premium users. e.g. Entertainment Services.
* A naive fix is telling the model in the system prompt "don't offer this to Non-Premium users"
* But this relies on the model choosing to follow an instruction, which might not strictly follow.

### The Fix: `wrap_model_call` — Making Entertainment Tool Disappear For Non-Premium users
Our Agent will get the Access to Tools while running, i.e. tools will get initialised dynamically while agent running.

#### 2 approaches depending on whether tools are known ahead of time
1. **Filtering pre-registered tools:**:
    * When all tools are known at agent creation time, you can pre-register them 
    * This approach is best when:
        * All possible tools are known at startup time
        * Want to filter based on permissions or conversation state
        * Tools are static but their availability is Dynamic
    * Dynamically filter which ones are exposed to the model based on
        * **State**: Enable advanced tools only after certain conversation milestones
        * **Preferences**: Filter tools based on user preferences in Store
        * **Runtime Context**: Filter tools based on user permissions from Runtime Context

2. **Runtime tool registration**:
    * When tools are discovered or created at runtime.
    * e.g., loaded from an MCP server, generated based on user data, or fetched from a remote registry, 
    * We need to both register the tools and handle their execution dynamically.
    > This will need a Middleware: How we can change things as things gets initialised.
    > How it can change things in middle of the execution
    
    This requires two middleware hooks:
    1. `wrap_model_call` - Add the dynamic tools to the request
    2. `wrap_tool_call` - Handle execution of the dynamically added tools

#### 1. Filtering pre-registered tools - Based on Conversation State
* We have defined tools to provide deals to the user `standard_deals` and `premium_deals`.
* `premium_deals` tool is only visible/called for premium users, normal users should only get `standard_deals`.
* We have defined and a `gate_premium_tools` middleware function to filter out the `premium_deals` tool for non-premium members. 
* Finallly, we have created an agent that uses these tools and middleware to provide context-aware tool calls based on the user's membership tier.
* Based on the state of the user, the agent will either show both standard and premium deals or only standard deals.

**Middleware `gate_premium_tools`**
* This is a middleware function that will filter out the `premium_deals` tool for non-premium members.
* `@wrap_model_call` allows to define middleware function that can intercept and modify the request/response of model call.
```python
    @wrap_model_call 
    def gate_premium_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        """Only expose premium_deals to premium members.
        Args:
        request (ModelRequest): The incoming request to the model, which includes the tools and state.
        handler (Callable[[ModelRequest], ModelResponse]): The next handler in the middleware chain, 
            which will process the request if the user is allowed to access the tool.
        """
```

In [87]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

class CustomAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    user_membership_tier: str  # <--- Our custom state field

@tool
def standard_deals(product: str) -> str:
    """Show standard deals for a product."""
    normal_price = 1200.0
    discounted_price = normal_price * 0.9
    return f"Normal price for {product} is ${normal_price:.2f}. Standard deal for {product} is 10% off. discounted price: ${discounted_price:.2f}"

@tool
def premium_deals(product: str) -> str:
    """Show premium deals for a product. Premium members only."""
    normal_price = 1200.0
    discounted_price = normal_price * 0.7
    return f"Normal price for {product} is ${normal_price:.2f}. Premium deal for {product} is 30% off. discounted price: ${discounted_price:.2f}"


@wrap_model_call 
def gate_premium_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose premium_deals to premium members."""
    membership_tier = request.state.get("user_membership_tier", 'guest')
    if membership_tier != "premium":
        allowed = [t for t in request.tools if t.name != "premium_deals"]
        request = request.override(tools=allowed)
    return handler(request)

gated_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[standard_deals, premium_deals],
    middleware=[gate_premium_tools],
    state_schema=CustomAgentState 
)

#### 1. Check with agent if there are any premium deals on Iphone 15 for Standard Users
* In this case only `standard_deals` deals should be visible.
* If model suggests a tool call it can only do that for `standard_deals` tool.
* This should not get any premium deals even if user speciifically asks in prompt.
* It should only get standard deals, all deals available for standard users are 10%.

In [100]:
result = gated_agent.invoke(
    {"messages": [HumanMessage(content="Is there any ongoing premium deal for the iPhone 15?")], "user_membership_tier": "non-premium"}
)
print("Non-premium member result:", result["messages"][-1].content)


Non-premium member result: Currently, there isn’t a separate “premium deal” listed for the iPhone 15. The available promotion is a standard deal: 10% off, bringing the price from $1200 to $1080.

If you’d like, I can:
- Monitor for any future premium promotions on the iPhone 15
- Look for any other bundled or trade-in options that might offer extra value
- Help you proceed with the current deal to secure the $1080 price


In [101]:
pretty_print_messages(result)

Message 1
Role : Human
--------------------------------------------------------------------------------
Is there any ongoing premium deal for the iPhone 15?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : standard_deals
    Args : {'product': 'iPhone 15'}
    ID   : call_XVg2cYhG2oEaHB8r4HBHnu0T
Message 3
Role : Tool
--------------------------------------------------------------------------------
Normal price for iPhone 15 is $1200.00. Standard deal for iPhone 15 is 10% off. discounted price: $1080.00

Tool Name : standard_deals
Tool Call : call_XVg2cYhG2oEaHB8r4HBHnu0T
Message 4
Role : AI
--------------------------------------------------------------------------------
Currently, there isn’t a separate “premium deal” listed for the iPhone 15. The available promotion is a standard deal: 10% off, bringing the price from $1200 to $1080.

If you’d like, I can:
- Monitor for any future premium promotions on the iP

#### 1. Check with agent if there are any premium deals on Iphone 15 for Premium Users
* In this case both `standard_deals` and `premium_deals` should be visible.
* Model can suggest a tool(s) to both tools or any one of them based on prompt mentioning about premium deals or standard deals.
* In our prompt we asked specifically for premium deals which have 30% off on items, so should get 30% discounted price.

In [102]:
result = gated_agent.invoke(
    {"messages": [HumanMessage(content="Is there any ongoing premium deal for the iPhone 15?")], "user_membership_tier": "premium"}
)
print("Premium member result:", result["messages"][-1].content)


Premium member result: Yes. There’s a premium deal for the iPhone 15:
- Regular price: $1200
- Premium discount: 30% off
- Discounted price: $840

Would you like me to apply this discount to a purchase, or view color/storage options and related bundles?


In [103]:
pretty_print_messages(result)

Message 1
Role : Human
--------------------------------------------------------------------------------
Is there any ongoing premium deal for the iPhone 15?
Message 2
Role : AI
--------------------------------------------------------------------------------


Tool Calls:
  • Name : premium_deals
    Args : {'product': 'iPhone 15'}
    ID   : call_T7axt5bDpWZVHzXi9Z5hIWNs
Message 3
Role : Tool
--------------------------------------------------------------------------------
Normal price for iPhone 15 is $1200.00. Premium deal for iPhone 15 is 30% off. discounted price: $840.00

Tool Name : premium_deals
Tool Call : call_T7axt5bDpWZVHzXi9Z5hIWNs
Message 4
Role : AI
--------------------------------------------------------------------------------
Yes. There’s a premium deal for the iPhone 15:
- Regular price: $1200
- Premium discount: 30% off
- Discounted price: $840

Would you like me to apply this discount to a purchase, or view color/storage options and related bundles?
